# Ranking Pages for Content Review: A Decision-Support Model Using Observed Search Visibility Signals

## Abstract

SEO and content teams cannot manually inspect every page in a large inventory for refresh opportunities, creating a need for systematic prioritization. This study builds a ranking-based decision-support system on an anonymized content-performance dataset of 331,437 aggregated pages across 55 clients, derived from FlyRank's pseudonymized warehouse release, using two observed March 2026 search-visibility features: impression volume and average search position. A Logistic Regression model validated with client-grouped out-of-sample cross-validation produces a ranked review queue that achieves a Precision@1000 of 0.971 ± 0.030 compared to 0.828 ± 0.119 for a transparent rule-based baseline, representing approximately 1.17× lift. The ranked review queue identifies where human SEO review is most likely to be productive, serving as a decision-support tool rather than an autonomous content optimizer. No causal claims are made regarding the effect of content changes on search performance.

---

## 1. Introduction

### The practical problem

A content portfolio of tens of thousands of pages cannot be reviewed manually at uniform depth. SEO and content teams face a prioritization problem: given limited review capacity, which pages deserve human inspection first? This is not an automation problem—editorial judgment, brand context, and strategic intent remain essential. It is a **triage** problem: how to route limited expert attention to the pages most likely to benefit from review.

### Why declining or underperforming content needs prioritization

Content that once performed well in search can lose visibility over time due to evolving search landscapes, competitor updates, or shifting user intent. Pages that show observable signals of underperformance—such as search impressions without corresponding clicks, or deep search rankings despite proven topic demand—are candidates for human review. Without a systematic ranking, teams either review pages in arbitrary order (wasting capacity on low-impact pages) or rely on simple rules that may miss non-obvious patterns in the data.

### What this work supports

This study builds and validates a **ranked review queue**: an ordered list of content pages, scored by a machine-learning model, that identifies where human SEO review is most likely to be productive. The output is a decision-support tool—not an autonomous content optimizer, not a quality classifier, and not a causal model of search ranking.

### What the model does and does NOT claim

- **Does:** Rank pages by observed search-visibility patterns, producing a prioritized queue for human inspection.
- **Does NOT:** Claim that refreshing a page will improve its search rankings.
- **Does NOT:** Reverse-engineer or predict Google's algorithm.
- **Does NOT:** Establish causal relationships between content changes and search performance.
- **Does NOT:** Replace human editorial judgment.

### FlyRank context

This work is developed within the FlyRank ML internship program, using FlyRank's pseudonymized content-performance dataset. All client names, domains, URLs, and private search queries have been removed. The analysis operates on aggregated, anonymized metrics only.

---

## 2. Data

### Source and structure

The modeling dataset consists of 331,437 aggregated page-level records across 55 pseudonymized clients, derived from the March 2026 partition of FlyRank's pseudonymized warehouse release (`FlyRank/internship-warehouse` on Hugging Face). Each record represents one pseudonymized content page within one pseudonymized client, with observed search and engagement metrics aggregated from approximately 9.8 million daily fact rows (`fact_content_daily_performance`).

### Relationship to the larger warehouse

The full warehouse release contains approximately 79 million daily fact rows (`fact_content_daily_performance`), 519,606 content items (`dim_content`), and 104 pseudonymized clients (`dim_clients`), covering daily search-performance data from January 2025 through June 2026. For this analysis, the March 2026 partition of the daily fact table was used directly, yielding 9,841,378 daily rows aggregated to 331,437 page-level records across 55 clients.

### Observation window

The analysis uses the March 2026 observation window. All features are derived from March 2026 data only. No April or May data is loaded or used, ensuring no future-window information enters the features.

### What was excluded and why

- **Client names, domains, URLs, page titles, keywords, and raw search queries:** Removed during pseudonymization to protect client privacy.
- **Product-decision flags** (`health_score`, `priority_score`, `action_type`, etc.): Intentionally excluded from the dataset so the model learns from observable signals rather than copying existing product decisions.
- **Rows with zero impressions:** These pages have no observable search presence and cannot be meaningfully evaluated for content refresh opportunities.
- **GA4-only features with low coverage:** Only ~4% of daily rows have GA4 data; these features were excluded to avoid noise from sparsely populated fields.

### Public-safety considerations

All identifiers are pseudonymized hashes (`content_` + 12 hex chars, `client_` + 10 hex chars). The notebook prints no client names, domains, URLs, raw queries, or titles. Figures use only aggregated metrics and pseudonymized IDs.

---

## 3. Methodology

### 3.1 Research assumptions

This study operates under the following assumptions:

1. **Observable search signals are informative.** Pages with search impressions, varying positions, and differing content characteristics exhibit patterns that can be distinguished by a learning algorithm.
2. **A ranked queue is more useful than a binary filter.** Rather than labeling every page as "needs review" or "does not need review," a continuous ranking allows teams to allocate review capacity proportional to model confidence.
3. **Client-grouped evaluation approximates real-world deployment.** In practice, a model would be applied to new clients not seen during training. GroupKFold validation, which holds out entire clients, approximates this scenario.
4. **The proxy label is a useful stand-in, not ground truth.** The evaluation label is constructed from observable metrics within the same observation window. It is not a direct measurement of whether a page genuinely needs human intervention.

### 3.2 Label definition

The proxy label is:

```
opportunity_proxy = (impressions > 0) & (clicks == 0)
```

A page is labeled proxy-positive when it has measurable search impressions but zero clicks in the observation window. This is interpreted as: the page is visible in search results but not generating click-through traffic, suggesting a potential content or metadata review opportunity.

**This is a proxy label, not a direct measurement of future organic performance.** The label is constructed from the same March 2026 window used for features. It identifies a current-state pattern (impressions without clicks), not a prediction of future decline. Describing this as "predicting which pages will decline" would be unsupported by the data structure.

**Positive-class base rate:** 32.6% of pages in the modeling dataset are proxy-positive.

### 3.3 Features

Two features are used, both derived from observed March 2026 data and available at decision time:

| Feature | Description | Rationale |
|---|---|---|
| `march_gsc_impressions` | Total Google Search Console impressions for the page across March 2026 | Pages with measurable search impression volume have active search exposure; this is the primary signal of search demand |
| `march_gsc_avg_position` | Mean GSC ranking position across March 2026 (days with position ≤ 0 excluded) | Pages ranking deeper in search results have different opportunity profiles than top-ranking pages; position is not a component of the proxy label, making it an independent signal |

All features are observable at decision time. No features are derived from the label, from future data, or from product-decision outputs.

**Note on feature count:** The Week 1–3 starter pipeline used five features (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`, `char_count`) on a 30,000-row teaching dataset. The final warehouse model narrows to two features on 331,437 pages because the warehouse data provides direct GSC metrics (impressions, position) that subsume the starter-pipeline engineered features, and a smaller feature set reduces overfitting risk in client-grouped cross-validation.

### 3.4 Baseline

The baseline is a transparent rule-based scoring system that combines position and impression signals into a single priority score:

```
position_score = 0   if avg_position <= 3
                 30  if avg_position 4–10
                 60  if avg_position 11–20
                 80  if avg_position 21–50
                 100 if avg_position > 50
                 0   if avg_position missing

impression_score = 0   if impressions = 0
                   10  if impressions 1–200
                   20  if impressions 201–1000
                   30  if impressions > 1000

baseline_score = position_score + impression_score
```

Pages are ranked by `baseline_score` descending. This baseline uses only two signals (position and impressions) with hand-tuned thresholds, representing what a human analyst might construct without machine learning. It was developed in Week 4 (`w04_baseline_score.ipynb`).

### 3.5 Model

The primary model is a **Logistic Regression classifier** (scikit-learn `LogisticRegression`, `max_iter=1000`, `random_state=42`), with features standardized via `StandardScaler` (fitted on training data only, missing position values filled with 0 after scaling).

Logistic Regression was selected as the primary model because:

1. It outputs calibrated probabilities suitable for ranking.
2. It was independently audited in Week 6 (`w06_validation_audit.ipynb`), confirming reproducibility across multiple training runs.
3. It achieves higher Precision@1000 than the secondary Random Forest comparison model.
4. It was the model used to generate the Week 7 ranked action queue.

A **Random Forest classifier** (100 estimators, `random_state=42`) was also evaluated as a secondary comparison model using the same GroupKFold structure. Random Forest does not require feature scaling and tests whether nonlinear modelling adds value over the linear baseline.

### 3.6 Validation design

The validation uses **GroupKFold cross-validation** (5 folds), grouped by `client_hash_id`.

**Why client grouping matters:** Pages from the same client share structural similarities (same industry, content strategy, technical setup). If pages from the same client appear in both training and test sets, the model can memorize client-specific patterns rather than learning generalizable signal. GroupKFold ensures that in each fold, no client appears in both training and test data.

The split structure across 5 folds with 55 clients:
- Each fold: ~44 clients for training, ~11 clients for testing
- Test set size: ~66,000 rows per fold
- Zero client overlap between train and test in every fold

This design approximates the real-world scenario where a model trained on existing clients is applied to new, unseen clients.

### 3.7 Leakage checks

A systematic leakage audit was conducted (documented in `w06_validation_audit.ipynb`). Key findings:

| Check | Result |
|---|---|
| Future-window inputs | **PASS** — All features use March 2026 data only; no April/May data loaded |
| Label-derived features | **PASS** — `march_gsc_clicks` and CTR excluded because they are inputs to the proxy |
| Product flags | **PASS** — No `health_score`, `priority_score`, or product-decision flags used |
| Target-derived fields | **PASS** — `trend_direction` and `trend_pct` excluded from features |
| Train/test separation | **PASS** — GroupKFold ensures zero client overlap |
| Preprocessing leakage | **PASS** — StandardScaler fitted on training fold only |

**Important limitation — same-window proxy label:** The proxy label is constructed from the same March 2026 observation window used for features. This means the label is not a clean future prediction target. The model identifies pages that *currently* match a specific search-visibility pattern (impressions without clicks), not pages that *will* decline in the future. This limitation must not be hidden: it constrains what the Precision@K metrics demonstrate about genuine predictive skill.

**Mechanical overlap:** The feature `impressions` is also part of the proxy definition (`impressions > 0`). This means Precision@K is partly influenced by the proxy construction. Pages with zero impressions automatically receive proxy=0, which is a structural feature of the evaluation definition, not evidence of model learning.

---

## 4. Results

### 4.1 Model vs. baseline comparison

All methods are evaluated on the same GroupKFold splits, using the same test rows and the same Precision@K metric. The table below reports out-of-client-fold performance:

| Method | P@100 | P@500 | P@1000 | P@5000 |
|---|---:|---:|---:|---:|
| Baseline (rule-based) | 0.746 ± 0.105 | 0.821 ± 0.124 | 0.828 ± 0.119 | 0.702 ± 0.156 |
| Logistic Regression (primary) | 0.992 ± 0.016 | 0.982 ± 0.025 | 0.971 ± 0.030 | 0.910 ± 0.053 |
| Random Forest (secondary) | 0.964 ± 0.024 | 0.951 ± 0.014 | 0.953 ± 0.011 | 0.948 ± 0.016 |

**Base rate:** 32.6% (the proportion of proxy-positive pages in the dataset).

The Logistic Regression achieves P@100 = 0.992, meaning approximately 99 of its top 100 ranked pages are proxy-positive under client-grouped out-of-sample evaluation. At the primary operating point of P@1000, the Logistic Regression achieves 0.971 compared to the baseline's 0.828, representing approximately **1.17× lift**.

The Random Forest achieves P@1000 = 0.953 ± 0.011, with lower cross-fold variance than the Logistic Regression. However, the Logistic Regression was independently audited in Week 6 and is the model used for the Week 7 action queue, making it the primary decision-support model.

**Important caveat:** The baseline P@1000 in the Week 4–7 warehouse analysis (0.828) differs from the 0.240 figure in the starter pipeline (`outputs/model_report.md`). The difference arises because the starter pipeline uses a different dataset (30,000 rows, 54.2% decline-label base rate) and a different label definition (`is_declining_label` derived from `trend_direction`). The 0.240 baseline P@50 is specific to that starter dataset and label. The Week 4–7 warehouse analysis uses the opportunity proxy (32.6% base rate) and produces different absolute numbers.

### 4.2 Out-of-sample P@1000

The Logistic Regression achieves an out-of-sample P@1000 of **0.971 ± 0.030**, meaning approximately 971 of the top 1000 ranked pages are proxy-positive under client-grouped evaluation.

Compared to the baseline's P@1000 of 0.828 ± 0.119, the Logistic Regression achieves a **1.17× lift** at the top-1000 operating point.

The secondary Random Forest model achieves P@1000 = 0.953 ± 0.011, with lower cross-fold variance but slightly lower absolute precision than the Logistic Regression at this operating point.

### 4.3 Queue overlap analysis

The top-1000 queues produced by the logistic regression model and the rule-based baseline share only **6 pages** (0.6% of each set). This near-zero overlap indicates that the ML model and the rule-based baseline identify fundamentally different pages as high-priority for review.

The baseline prioritizes pages by position depth and impression volume using fixed thresholds. The logistic regression learns a continuous scoring function that weights these signals differently, producing a substantially different ranking.

In [1]:
# --- Data loading and feature computation ---
# This cell loads the March 2026 warehouse partition, computes page-level features,
# defines the opportunity proxy, and prepares data for modeling.

from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is not set."

from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)
print(f"March 2026 daily rows: {len(march_df):,}")

# Aggregate to page level
page_features = (
    march_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_gsc_impressions=("gsc_impressions", "sum"),
        march_gsc_clicks=("gsc_clicks", "sum"),
        _pos_count=("gsc_avg_position", "count"),
        _pos_sum=("gsc_avg_position", lambda x: x[x > 0].sum()),
        _pos_valid_n=("gsc_avg_position", lambda x: (x > 0).sum()),
    )
)

page_features["march_gsc_avg_position"] = np.where(
    page_features["_pos_valid_n"] > 0,
    page_features["_pos_sum"] / page_features["_pos_valid_n"],
    np.nan,
)
page_features = page_features.drop(columns=["_pos_count", "_pos_sum", "_pos_valid_n"])

# Define opportunity proxy label
page_features["opportunity_proxy"] = (
    (page_features["march_gsc_impressions"] > 0)
    & (page_features["march_gsc_clicks"] == 0)
).astype(int)

print(f"Page-level records: {len(page_features):,}")
print(f"Unique clients: {page_features['client_hash_id'].nunique()}")
print(f"Positive-class base rate: {page_features['opportunity_proxy'].mean():.1%}")
print(f"Pages with position data: {page_features['march_gsc_avg_position'].notna().mean():.1%}")

d:\download_99\Anaconda\envs\Machine_Learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


March 2026 daily rows: 9,841,378
Page-level records: 331,437
Unique clients: 55
Positive-class base rate: 32.6%
Pages with position data: 52.9%


### Model lineage

The results reported in Section 4 trace to the following reproducible pipeline:

| Week | Notebook | What happened |
|---|---|---|
| W05 | `w05_model.ipynb` | Trained Baseline, LR, and RF with GroupKFold (5 folds, client-grouped). Established the model comparison table. |
| W06 | `w06_validation_audit.ipynb` | Independently validated LR and baseline. Confirmed reproducible baseline results. LR remained consistent. |
| W07 | `w07_action_playbook.ipynb` | Trained LR on full March 2026 data for ranking. Generated the action queue. Produced practical recommendations. |
| W08 | `capstone.ipynb` | Documents and consolidates W05–W07 evidence. Does not constitute a new ML experiment. |

**Primary model:** Logistic Regression — carried through W05 training → W06 validation → W07 action queue.

**Secondary model:** Random Forest — evaluated in W05 as a comparison; lower mean P@1000 than LR but lower cross-fold variance.

The capstone does not retrain models. The comparison table in Section 4 reflects the W05/W06 validated outputs.

### 4.4 Feature importance

**Logistic Regression coefficients (scaled features):**

The Logistic Regression model produces interpretable coefficients on standardized features:

- `march_gsc_impressions`: −4.912 (negative due to fillna(0) preprocessing; within the impressions>0 subset, higher impressions raise probability)
- `march_gsc_avg_position`: +2.815 (deeper positions increase predicted probability, independent signal confirmed by signal audit)

**Random Forest permutation importance (secondary comparison):**

| Feature | Permutation Importance |
|---|---:|
| `march_gsc_impressions` | 0.470 ± 0.001 |
| `march_gsc_avg_position` | 0.316 ± 0.001 |

Both features carry importance in both models. The impressions feature shows higher importance due to its mechanical relationship with the proxy (impressions > 0 is part of the proxy definition). The position feature's importance represents genuinely independent signal.

### 4.5 Queue composition

The Week 7 action playbook generates the ranked review queue using the Logistic Regression model trained on the full March 2026 dataset. The LR score produces the ranked review queue:

- **Total pages scored:** 331,437
- **Top-1000 REVIEW tier:** 1,000 pages
- **MONITOR tier:** 174,304 pages (52.6%)
- **DEPRIORITISE tier:** 156,133 pages (47.1%)

Behavioral archetypes in the full dataset:

| Archetype | Count | Share |
|---|---:|---:|
| Ghost pages (impressions = 0) | 154,699 | 46.7% |
| Already performing (position ≤ 10, clicks > 0) | 96,782 | 29.2% |
| Hidden gems (impressions > 0, clicks = 0, position > 10) | 56,049 | 16.9% |
| Emerging pages (impressions 1–200, position 11–50) | 21,867 | 6.6% |
| Data gaps (position missing, impressions > 0) | 1,434 | 0.4% |
| Deep rankers (position > 50, impressions > 0) | 606 | 0.2% |

---

## 5. Results Visualizations

### 5.1 Figures

The following figures were generated in `w07_action_playbook.ipynb` and are saved to `work/figures/`:

| Figure | File | Description |
|---|---|---|
| Action tier distribution | `work/figures/action_distribution.png` | Pages by REVIEW / MONITOR / DEPRIORITISE tier |
| Archetype breakdown | `work/figures/archetype_breakdown.png` | Behavioral archetypes across the full dataset |
| Client concentration | `work/figures/client_concentration.png` | Client distribution in the top-1000 queue |
| Confidence distribution | `work/figures/confidence_distribution.png` | Data-signal confidence tiers |
| Queue vs. baseline overlap | `work/figures/queue_vs_baseline_overlap.png` | Pages shared between LR and baseline top-1000 queues (6 pages / 0.6%) |

These figures are referenced here rather than regenerated, consistent with the capstone's role as a documentation notebook that consolidates W05–W07 evidence.

---

## 6. Limitations & Honest Framing

### 6.1 Proxy label limitation

The evaluation target (`opportunity_proxy = (impressions > 0) & (clicks == 0)`) is constructed from the same March 2026 observation window used for features. It is **not** a clean future prediction target. The model identifies pages that currently match a specific search-visibility pattern, not pages that will decline in the future. Describing this as "predicting content decay" or "forecasting ranking declines" would be unsupported by the data structure.

### 6.2 Mechanical overlap limitation

The feature `impressions` is mechanically related to the proxy definition (impressions > 0 is part of the proxy). This means Precision@K is partly influenced by the evaluation construction. Pages with zero impressions automatically receive proxy=0, which is a structural feature of the definition, not evidence of model learning. The independent signal comes from the `position` feature, which is not a component of the proxy.

### 6.3 Generalization limitation

The client-grouped evaluation (GroupKFold with 55 clients) tests generalization across client groups within a single portfolio. The dataset represents one business context (FlyRank's content portfolio). Results may not generalize to different industries, content types, or search landscapes. A 5-fold grouped split is a reasonable starting point but not a confirmation of deployment performance.

### 6.4 Observational data limitation

The analysis identifies associations and patterns in observed search data. It does not establish causality. We observe that pages with certain position and impression profiles are more likely to be proxy-positive; we do not claim that changing a page's position would alter its proxy status. Only controlled experiments (A/B tests, randomized interventions) can support causal claims.

### 6.5 Decision-support limitation

The output is a ranked review queue. It tells a human reviewer which pages to inspect first. It does not prescribe specific actions (refresh, rewrite, merge, prune). Human SEO and content judgment remains necessary for every editorial decision. The model scores where to look; the human decides what to do.

### 6.6 Single-window limitation

Only March 2026 data is used. No seasonal, trend, or longitudinal patterns are captured. The analysis is a cross-sectional snapshot of one month's search visibility. The results reflect March 2026 conditions specifically.

### 6.7 Client concentration limitation

One client dominates ~50% of the top-1000 queue. The queue's composition reflects this client's feature distribution. This is not a validation bug—GroupKFold ensures this client is never in both train and test—but it is a practical consideration for queue deployment.

### 6.8 RF downstream validation coverage

The Random Forest was evaluated in Week 5 using the same GroupKFold structure, achieving P@1000 = 0.953 ± 0.011 with lower cross-fold variance than the Logistic Regression. However, the Random Forest was not independently audited in Week 6, was not used to generate the Week 7 action queue, and therefore has less downstream validation coverage than the Logistic Regression. It remains a useful secondary comparison model.

---

## 7. Ranked Recommendations / Action Playbook

### How the ranking translates to actions

The Logistic Regression score produces the ranked review queue. The playbook translates this into a prioritized review queue:

```
LR score → ranked queue → reason codes + archetype → human review action
```

The queue prioritizes inspection; it does not automatically prescribe a content change.

### Action tiers

| Tier | Definition | Count |
|---|---|---:|
| **REVIEW** | Top 1,000 ranked pages — primary human-review queue | 1,000 |
| **MONITOR** | Pages with observable search signal but outside the primary review queue | ~174,000 |
| **DEPRIORITISE** | Pages with insufficient observable signal | ~156,000 |

### Behavioral archetypes and suggested review directions

| Archetype | What it means | Review direction |
|---|---|---|
| **Hidden gems** | Impressions > 0, clicks = 0, position > 10 | Investigate content alignment, title/meta improvements |
| **Deep rankers** | Position > 50, impressions > 0 | Investigate competitive query targeting, structural SEO |
| **Emerging pages** | Low impressions, position 11–50 | Monitor for growth; verify topic targeting |
| **Already performing** | Position ≤ 10, clicks > 0 | Low priority for refresh; monitor for drift |
| **Ghost pages** | Impressions = 0 | Check indexing, crawlability, technical issues |
| **Data gaps** | Position missing, impressions > 0 | Investigate position data availability |

### Why the ML queue differs from the baseline

The baseline uses fixed thresholds on position and impressions. The Logistic Regression learns a continuous scoring function that combines these signals, producing a substantially different ranking. The 0.6% overlap between the top-1000 queues confirms that the two methods identify nearly disjoint sets of pages as highest priority.

### Actions that require human judgment

The following actions are outside the model's evidence scope and require human SEO judgment:
- Publishing or deleting content
- Merging pages
- Rewriting content or changing title/meta tags
- Technical SEO changes
- Claiming that refreshing a page will improve rankings (no causal evidence)

---

## 8. Reproducibility

### How to rerun this analysis

1. **Repository:** This capstone notebook is located at `work/notebooks/capstone.ipynb`.
2. **Data access:** The warehouse data is hosted at `FlyRank/internship-warehouse` on Hugging Face (gated; request access and accept data-use terms). The notebook loads the March 2026 partition directly.
3. **Dependencies:** Python 3.12+, pandas, numpy, scikit-learn, matplotlib, huggingface-hub, python-dotenv.
4. **Environment:** Set `HF_TOKEN` in a `.env` file at the repository root.
5. **Execution:** Run the notebook top-to-bottom (Runtime → Run all).

### Weekly pipeline

| Week | Notebook | Purpose |
|---|---|---|
| Week 5 | `w05_model.ipynb` | Model training and comparison (LR, RF, baseline) |
| Week 6 | `w06_validation_audit.ipynb` | Independent validation and research-claim audit |
| Week 7 | `w07_action_playbook.ipynb` | Action playbook and LR-ranked queue generation |
| Week 8 | `capstone.ipynb` | Capstone paper and final presentation |

The final recommendations trace back to the validated Logistic Regression workflow from Week 5, independently audited in Week 6, and operationalized in Week 7.

### Supporting notebooks

| Notebook | Purpose |
|---|---|
| `w01_research_question.ipynb` | Research question framing |
| `w02_ml_task_framing.ipynb` | ML task definition |
| `w03_data_contract.ipynb` | Data contract and access setup |
| `w04_signal_audit.ipynb` | Signal audit with verdicts |
| `w04_baseline_score.ipynb` | Baseline rule construction |
| `w05_model.ipynb` | Model training and comparison |
| `w06_validation_audit.ipynb` | Validation and leakage audit |
| `w07_action_playbook.ipynb` | Action playbook and queue generation |

### Generated artifacts

| Artifact | Location |
|---|---|
| Action distribution figure | `work/figures/action_distribution.png` |
| Archetype breakdown figure | `work/figures/archetype_breakdown.png` |
| Client concentration figure | `work/figures/client_concentration.png` |
| Confidence distribution figure | `work/figures/confidence_distribution.png` |
| Queue vs. baseline overlap figure | `work/figures/queue_vs_baseline_overlap.png` |
| Starter pipeline model report | `outputs/model_report.md` |
| ML action queue | `work/outputs/ml_action_queue.csv` |

### Data distinction

- **Gated warehouse access:** The full ~79M-row warehouse release requires Hugging Face authentication. It is not committed to the repository.
- **No credentials committed:** The `.env` file containing `HF_TOKEN` is gitignored.

---

## 9. Acknowledgments & Data Credit

**Built on the [FlyRank](https://flyrank.ai/) ML Internship dataset.**

The pseudonymized warehouse release (`FlyRank/internship-warehouse` on Hugging Face) was prepared by FlyRank for internship use. All client names, domains, URLs, raw search queries, and private identifiers were removed or pseudonymized before release.

---

## 10. Week 8 — Tell the Story

### 5-Minute Demo Outline

1. **Question:** SEO teams cannot review every page manually. How do we rank 330,000+ pages so the right ones get human attention first?

2. **Data:** An anonymized dataset of 331,437 aggregated pages across 55 clients from FlyRank's pseudonymized warehouse release, using observed March 2026 search-visibility signals (impressions, position). No client names, URLs, or private queries.

3. **Method:** A transparent rule-based baseline (position + impression thresholds) vs. a Logistic Regression model, validated with client-grouped cross-validation (no client appears in both train and test). Random Forest included as a secondary comparison.

4. **One key chart:** The queue-vs-baseline overlap figure — showing that the ML model and the baseline identify nearly disjoint sets of pages as highest priority (6 pages / 0.6% overlap in the top 1000).

5. **One honest result:** Under client-grouped evaluation, the Logistic Regression achieves P@1000 of 0.971 ± 0.030 compared to the baseline's 0.828 ± 0.119 (1.17× lift). The model concentrates proxy-positive pages more effectively, but the proxy label is a same-window construct, not a future prediction.

6. **One recommendation:** Use the ranked queue as a review triage tool. The model identifies where to look; the human decides what to do. Do not claim that refreshing a page will improve rankings without running an experiment.

7. **Closing takeaway:** Machine learning can produce a meaningfully better review queue than simple rules, but the output is decision-support, not automation. Honest validation and transparent limitations are what make the result trustworthy.

---

### Social Post

Built a decision-support model for content review prioritization using FlyRank's anonymized search-performance data (331,437 pages, 55 clients). A Logistic Regression trained on two search-visibility features — impression volume and search position — produces a ranked review queue that achieves 1.17× lift over a rule-based baseline under client-grouped cross-validation. Key insight: the ML model and the baseline identify nearly different pages as highest priority (0.6% top-1000 overlap), suggesting the learned ranking captures non-obvious patterns in search visibility. The output is a triage tool, not an automation system — human SEO judgment remains essential. #MachineLearning #SEO #ContentStrategy #FlyRank

---

### Employer-Facing Summary

Built a Logistic Regression ranking model that produces a prioritized content review queue from anonymized search-visibility data (331,437 pages across 55 clients), validated using client-grouped cross-validation to test generalization to unseen clients. The model achieves out-of-sample P@1000 of 0.971 ± 0.030 compared to a 0.828 ± 0.119 baseline (1.17× lift), with the Week 7 ranked action queue tracing directly back to the validated LR workflow. The work demonstrates ML engineering skills in feature design, honest validation, leakage auditing, and producing a reproducible, public-safe research artifact.

## Self-check

Before submission, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.